=== Resume cell ===

In [1]:
# Setup for the spatial slice below: re-runs the notebook cells it depends on (found by their first line),
# so it works after a kernel restart. Cell 6 is deliberately NOT run: it overwrites the hand-labelled
# boilerplate_precision_check.csv. Cells that only print or log duplicate rows are skipped too.
import json as _json
from pathlib import Path as _Path

_nb_path = next(p for p in [_Path("eda.ipynb"), _Path("notebooks/eda.ipynb")] if p.exists())
_cells = _json.loads(_nb_path.read_text(encoding="utf-8"))["cells"]


def _run(marker):
    """Execute the code cell whose source starts with `marker` in the notebook namespace."""
    src = next("".join(c["source"]) for c in _cells
               if c["cell_type"] == "code" and "".join(c["source"]).startswith(marker))
    print(f">> {marker.splitlines()[0][:70]}")
    exec(compile(src, f"<setup: {marker[:20]}>", "exec"), globals())


for _m in [
    "import warnings\nfrom pathlib import Path",           # paths + Kaggle path (cached download)
    "# Load all regional files",                             # df
    "# Text cleaning",                                       # clean_text
    "# Word count as a flag",                                # is_too_short
    "# Boilerplate detection on RAW text",                   # is_boilerplate
    "# Exact-duplicate detection",                           # has_text, is_duplicate
    "import numpy as np\nimport pandas as pd\n\n# Fixed sample",  # df_sample, embeddings, gold
    "from scipy.stats import entropy",                       # evaluate(), NON_TOPICAL
    "import json\nfrom datetime import datetime",           # SEED, log_run
    "import itertools\nimport json\nimport time",          # REPR, EXP_*, get_umap, ...
]:
    _run(_m)
print("Setup done.")


>> import warnings
>> # Load all regional files
Total files: 123
Total records: 615000
post_id unique: True
Columns: ['raw_text', 'coordinates', 'region', 'post_id', 'lon', 'lat']
>> # Text cleaning
URL leftovers: 4
@ leftovers: 49
>> # Word count as a flag
count    615000.000000
mean         10.570603
std          11.207545
min           0.000000
25%           3.000000
50%           7.000000
75%          14.000000
max          91.000000
Name: word_count, dtype: float64
Too short (<4 words): 189998
>> # Boilerplate detection on RAW text
Total boilerplate: 15043 (2.4%)
>> # Exact-duplicate detection
Posts without text (emoji/punctuation only): 52069
Eligible posts: 409922
Exact duplicates: 3923 (1.0% of eligible)
dup_key
thank you so much                                                                    54
जय जय शर रम                                                                          43
me sigam que tbm sigo obrigado somos fortes bolsonaro reeleito em primeiro turno     38
जय शर 

In [12]:
SPATIAL_POOL_PATH = DATA_SAMPLE / "spatial_pool.csv"   # all non-trivial posts with coordinates; duplicates flagged, not dropped

## Spatial feasibility slice: choosing cities

The 100k sample is global, so it is probably too sparse inside any single city. Before spatial analysis, we need to know where posts are dense enough for a grid of H3 cells (resolution 8-9, roughly 0.5 km and 0.2 km edge).

Plan:
1. Save the full pool (same filter as the 100k sample) with coordinates, so we do not depend on the in-memory `df`.
2. Assign every post to H3 cells and rank coarse cells (resolution 4, about 20 km edge, roughly a metro area) by post count.
3. For each candidate, check how many posts fall into each fine cell. A city is usable only if most posts sit in cells with enough neighbours (we use 10 as a rough threshold, not a statistical rule).

Nothing is dropped or embedded here. Boilerplate stays flagged.

In [13]:
# Spatial pool: all non-trivial posts with coordinates, used for the spatial part
# (not to be confused with `pool` in the sampling cell, which is unique posts only).
# Duplicates stay IN the pool, only flagged (same convention as boilerplate):
# the default analysis uses `~is_duplicate`; the spatial part can optionally bring duplicates back
# (e.g. count each distinct text once per H3 cell) as a sensitivity check.
# Cached to disk; skipped if the cache exists. On the first run `df` must be in memory (cells 2-10 executed).
SPATIAL_COLS = ["post_id", "region", "lat", "lon", "clean_text", "dup_key", "is_boilerplate", "is_duplicate"]

if SPATIAL_POOL_PATH.exists():
    spatial_pool = pd.read_csv(
        SPATIAL_POOL_PATH,
        dtype={"post_id": str, "region": str, "clean_text": str, "dup_key": str},
        keep_default_na=False,
    )
    for col in ["is_boilerplate", "is_duplicate"]:
        spatial_pool[col] = spatial_pool[col].astype(str).str.lower().eq("true")
else:
    spatial_mask = ~df["is_too_short"] & df["has_text"]       # duplicates are NOT dropped here
    spatial_pool = df.loc[spatial_mask, SPATIAL_COLS].reset_index(drop=True)
    spatial_pool.to_csv(SPATIAL_POOL_PATH, index=False, encoding="utf-8")

assert spatial_pool["post_id"].is_unique
assert df_sample["post_id"].isin(spatial_pool["post_id"]).all(), "sample is not a subset of the spatial pool"
print("Spatial pool incl. duplicates:", len(spatial_pool), "| duplicates flagged:", spatial_pool["is_duplicate"].sum())
print("Without duplicates:", (~spatial_pool["is_duplicate"]).sum(), "| sample:", len(df_sample))  # expected: 420952

Spatial pool incl. duplicates: 424875 | duplicates flagged: 3923
Without duplicates: 420952 | sample: 100000


In [15]:
# H3 indexing. Requires: pip install "h3>=4"
import h3

cells9 = [h3.latlng_to_cell(la, lo, 9) for la, lo in zip(spatial_pool["lat"], spatial_pool["lon"])]
spatial_pool["h3_r9"] = cells9
spatial_pool["h3_r8"] = [h3.cell_to_parent(c, 8) for c in cells9]
spatial_pool["h3_r4"] = [h3.cell_to_parent(c, 4) for c in cells9]
spatial_pool["in_sample"] = spatial_pool["post_id"].isin(df_sample["post_id"])

# Default view: unique texts only (duplicates are brought back only for the sensitivity check)
sp_unique = spatial_pool[~spatial_pool["is_duplicate"]]

cand = (sp_unique.groupby("h3_r4")
        .agg(n_unique=("post_id", "size"), n_sample=("in_sample", "sum"))
        .sort_values("n_unique", ascending=False)
        .head(20))
centroids = [h3.cell_to_latlng(c) for c in cand.index]
cand["lat"] = [round(c[0], 3) for c in centroids]
cand["lon"] = [round(c[1], 3) for c in centroids]

top_sub = sp_unique[sp_unique["h3_r4"].isin(cand.index)]
cand["region"] = top_sub.groupby("h3_r4")["region"].agg(lambda s: s.mode().iat[0]).loc[cand.index].values

print(cand.to_string())

                 n_unique  n_sample     lat      lon region
h3_r4                                                      
8428de9ffffffff      2410       579  49.100 -123.138    102
8428341ffffffff      1932       455  37.098 -122.134     85
8419091ffffffff      1767       481  56.078   -4.895    118
848e601ffffffff      1677       396 -11.895  -75.734    111
84be631ffffffff      1595       399 -37.954  144.672     75
8426963ffffffff      1585       359  40.402 -111.831    112
848a8ddffffffff      1554       351  -2.652  -58.627     88
848046dffffffff      1432       351  -1.208  -47.780     60
849763bffffffff      1390       340 -22.327   23.740    109
841e41dffffffff      1358       351  48.409   31.062     65
844252dffffffff      1248       271  26.186   68.900    104
8480129ffffffff      1246       308  -3.837  -38.059     93
8439423ffffffff      1235       287  39.884    2.809    120
841f051ffffffff      1223       289  55.336   12.790    119
84b2c55ffffffff      1221       295 -33.

In [17]:
# Density inside each candidate: how many posts per fine cell?
# "share_in_10+" = share of posts that sit in cells with at least 10 posts (rough usability threshold)
def density_summary(sub, cell_col):
    per_cell = sub.groupby(cell_col).size()
    return pd.Series({
        "cells": len(per_cell),
        "median_posts": per_cell.median(),
        "share_in_10+": per_cell[per_cell >= 10].sum() / per_cell.sum(),
    })

rows = []
for cell in cand.index:
    sub = sp_unique[sp_unique["h3_r4"] == cell]
    r8 = density_summary(sub, "h3_r8").add_prefix("r8_")
    r9 = density_summary(sub, "h3_r9").add_prefix("r9_")
    rows.append(pd.concat([r8, r9]).rename(cell))

cand_report = cand.join(pd.DataFrame(rows))
print(cand_report.round(2).to_string())

                 n_unique  n_sample    lat     lon region  r8_cells  r8_median_posts  r8_share_in_10+  r9_cells  r9_median_posts  r9_share_in_10+
h3_r4                                                                                                                                            
8428de9ffffffff      2410       579  49.10 -123.14    102       1.0           2410.0              1.0       1.0           2410.0              1.0
8428341ffffffff      1932       455  37.10 -122.13     85       1.0           1932.0              1.0       1.0           1932.0              1.0
8419091ffffffff      1767       481  56.08   -4.89    118       1.0           1767.0              1.0       1.0           1767.0              1.0
848e601ffffffff      1677       396 -11.90  -75.73    111       1.0           1677.0              1.0       1.0           1677.0              1.0
84be631ffffffff      1595       399 -37.95  144.67     75       1.0           1595.0              1.0       1.0           15

### Reading the candidate table
- `n_pool` is the number of posts available in the city, `n_sample` how many already have embeddings.
- A good city has a high `n_pool` and a high `r8_share_in_10+`. If it is low at r8, we may have to use a coarser grid.
- Cities from different regions are preferred, so the result does not depend on one local dataset.
- The 10-post threshold is a heuristic for choosing cities, not a statistical criterion.

In [18]:
# How precise are the coordinates? Count distinct (lat, lon) pairs overall and per region
coords = spatial_pool[["region", "lat", "lon"]]
print("Distinct (lat, lon) pairs:", len(coords[["lat", "lon"]].drop_duplicates()), "of", len(coords), "posts")

pairs_per_region = coords.drop_duplicates().groupby("region").size()
print("Regions:", len(pairs_per_region))
print("Regions with a single coordinate pair:", (pairs_per_region == 1).sum())
print(pairs_per_region.describe())
print("Posts per region:")
print(coords.groupby("region").size().describe())

Distinct (lat, lon) pairs: 11598 of 424875 posts
Regions: 123
Regions with a single coordinate pair: 0
count    123.000000
mean      94.292683
std       44.034214
min       17.000000
25%       64.500000
50%       87.000000
75%      118.000000
max      206.000000
dtype: float64
Posts per region:
count     123.000000
mean     3454.268293
std       550.277472
min      1078.000000
25%      3376.000000
50%      3630.000000
75%      3747.500000
max      3897.000000
dtype: float64


In [19]:
# How far apart are posts within one region? (bounding box in km)
g = spatial_pool.groupby("region")
lat_km = (g["lat"].max() - g["lat"].min()) * 111
lon_km = (g["lon"].max() - g["lon"].min()) * 111 * np.cos(np.radians(g["lat"].mean()))
print(pd.DataFrame({"lat_km": lat_km, "lon_km": lon_km}).describe().round(3))

# Distinct H3 cells per region at several resolutions
for res in (6, 7, 8, 9):
    n_cells = spatial_pool.groupby("region")["h3_r9"].apply(
        lambda s: s.map(lambda c: h3.cell_to_parent(c, res)).nunique())
    print(f"res {res}: median distinct cells per region = {n_cells.median():.0f}, max = {n_cells.max()}")

         lat_km     lon_km
count   123.000    123.000
mean   1193.404   2003.688
std     778.576   4839.288
min     240.417    210.291
25%     718.849    795.420
50%    1014.902   1018.898
75%    1399.506   1327.320
max    4569.041  38100.472
res 6: median distinct cells per region = 87, max = 206
res 7: median distinct cells per region = 87, max = 206
res 8: median distinct cells per region = 87, max = 206
res 9: median distinct cells per region = 87, max = 206


In [20]:
# Unit of analysis: a distinct coordinate pair = a "place" (city-level point).
# How many (unique-text) posts does each place have?
place_counts = sp_unique.groupby(["lat", "lon"]).size()
print("Places (distinct coordinates):", len(place_counts))
print(place_counts.describe().round(1))
for t in (10, 30, 50, 100, 300):
    sel = place_counts[place_counts >= t]
    print(f"places with >= {t:>3} posts: {len(sel):>5} | cover {sel.sum() / place_counts.sum():.1%} of posts")

Places (distinct coordinates): 11591
count    11591.0
mean        36.3
std        105.0
min          1.0
25%          2.0
50%          7.0
75%         24.0
max       2410.0
dtype: float64
places with >=  10 posts:  4921 | cover 94.8% of posts
places with >=  30 posts:  2547 | cover 85.1% of posts
places with >=  50 posts:  1771 | cover 78.1% of posts
places with >= 100 posts:   989 | cover 64.9% of posts
places with >= 300 posts:   276 | cover 37.0% of posts


### Dead end ;(